<a href="https://colab.research.google.com/github/amit-sahu-a11y/ML_projects_for_practice/blob/main/Professional_Ticket_Classification_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Support Ticket Classification

### Objective
Build a machine learning model that automatically classifies support tickets into:
- Billing
- Technical
- HR
- General

**Approach:** TF-IDF + Multinomial Naive Bayes

> *Note:* I started with a simple baseline model because Naive Bayes is fast and usually performs well for text classification tasks.


In [1]:


import pandas as pd
import numpy as np
import re
import string
import pickle

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)


## Load Dataset

In [2]:
df = pd.read_csv("tickets_large.csv")

print("Shape:", df.shape)
df.head()


Shape: (200, 3)


,subject,body,category
0,Office address,Share office location. Ticket reference 1024.,General
1,Office address,Share office location. Ticket reference 1015.,General
2,Invoice missing,Please send my latest invoice. Ticket referenc...,Billing
3,Interview schedule,When is my interview? Ticket reference 1040.,HR
4,Server error,Receiving 500 internal server error. Ticket re...,Technical


In [3]:

df.info()

# print("\nMissing Values")
df.isnull().sum()

# print("\nCategory Distribution")
df["category"].value_counts()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   200 non-null    object
 1   body      200 non-null    object
 2   category  200 non-null    object
dtypes: object(3)
memory usage: 4.8+ KB


,count
category,
General,50
Billing,50
HR,50
Technical,50


checking duplicate values

In [12]:
df.duplicated().sum()

np.int64(0)

there's no duplicate values

### Observation

The dataset looks clean and balanced across all four categories,
so I don't need any additional handling for class imbalance.


## Text Preprocessing

In [4]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    text = re.sub(r"\d+", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


df["text"] = (
    df["subject"] + " " + df["body"]
)

df["text"] = df["text"].apply(clean_text)

df[["text","category"]].head()


,text,category
0,office address share office location ticket re...,General
1,office address share office location ticket re...,General
2,invoice missing please send my latest invoice ...,Billing
3,interview schedule when is my interview ticket...,HR
4,server error receiving internal server error t...,Technical


In [16]:
df[["text","category"]].tail()

,text,category
195,website down website is not working ticket ref...,Technical
196,payment pending payment is still pending ticke...,Billing
197,password reset reset link has expired ticket r...,Technical
198,app crash the mobile app crashes immediately t...,Technical
199,training do you provide product training ticke...,General


I combined the subject and body because both contain useful information for prediction.


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["category"],
    test_size=0.2,
    random_state=42,
    stratify=df["category"]
)

print("Training:", len(X_train))
print("Testing :", len(X_test))


Training: 160
Testing : 40


## Build Model

In [6]:
ticket_model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("classifier", MultinomialNB())
])

ticket_model.fit(X_train, y_train)

print("Model training completed.")


Model training completed.


In [7]:
predictions = ticket_model.predict(X_test)

print("Accuracy:", round(
    accuracy_score(y_test, predictions),3
))

print("\nClassification Report")
print(classification_report(
    y_test,
    predictions
))

print("\nConfusion Matrix")
print(confusion_matrix(
    y_test,
    predictions
))


Accuracy: 1.0

Classification Report
              precision    recall  f1-score   support

     Billing       1.00      1.00      1.00        10
     General       1.00      1.00      1.00        10
          HR       1.00      1.00      1.00        10
   Technical       1.00      1.00      1.00        10

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40


Confusion Matrix
[[10  0  0  0]
 [ 0 10  0  0]
 [ 0  0 10  0]
 [ 0  0  0 10]]


## Prediction Function

In [11]:
urgent_words = [
    "urgent",
    "down",
    "critical",
    "failed",
    "not working",
    "immediately",
    "error",
]

def predict_ticket(ticket):

    cleaned = clean_text(ticket)

    prediction = ticket_model.predict([cleaned])[0]

    probabilities = ticket_model.predict_proba([cleaned])[0]

    confidence = probabilities.max()

    priority = "Urgent" if any(
        word in cleaned for word in urgent_words
    ) else "Normal"

    review = (
        "Needs Human Review"
        if confidence < 0.60
        else "Auto Assigned"
    )

    print("="*50)
    print("Ticket :", ticket)
    print("Category:", prediction)
    print("Confidence:", f"{confidence:.2%}")
    print("Priority:", priority)
    print("Status:", review)


In [9]:
sample_tickets = [

    "My payment failed and I was charged twice.",

    "Website is down and showing server error.",

    "Need my salary slip for this month.",

    "Can you share office timing?",

    "Password reset link is not working."

]

for ticket in sample_tickets:
    predict_ticket(ticket)


Ticket : My payment failed and I was charged twice.
Category: Billing
Confidence: 81.24%
Priority: Urgent
Status: Auto Assigned
Ticket : Website is down and showing server error.
Category: Technical
Confidence: 80.73%
Priority: Urgent
Status: Auto Assigned
Ticket : Need my salary slip for this month.
Category: HR
Confidence: 70.00%
Priority: Normal
Status: Auto Assigned
Ticket : Can you share office timing?
Category: General
Confidence: 66.27%
Priority: Normal
Status: Auto Assigned
Ticket : Password reset link is not working.
Category: Technical
Confidence: 76.77%
Priority: Urgent
Status: Auto Assigned


## Save Model

Saving the trained pipeline so it can be reused later without retraining.


In [10]:
with open("ticket_classifier.pkl","wb") as file:
    pickle.dump(ticket_model,file)

print("Model saved successfully.")


Model saved successfully.


# Reflection

- I started with Multinomial Naive Bayes because it is simple and effective for TF-IDF features.
- The dataset is synthetic, so the model may perform better than it would on real customer tickets.
- Given more real-world data, I would compare Logistic Regression and Linear SVM.
- In the future I would also build a Streamlit interface and continuously retrain the model using new support tickets.
